# Libraries
NOTE: After installing pip requirements, restart the kernel before continuing, to ensure that all installed/upgraded packages are reimported freshly.

(Fixes an issue with `typing_extensions`: https://github.com/python/typing_extensions/issues/510)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install torch==2.9.0
!pip install torchaudio==2.9.0 torchvision==0.24.0
!pip install wandb datasets torch_geometric

In [ ]:
# Import unsloth before everything else
from unsloth import FastLanguageModel
import os
import torch
import re
import numpy as np
from typing import Tuple, List, Dict
import networkx as nx
from collections import defaultdict
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from tqdm import tqdm
import json

# Load TA and DMU Instances

In [ ]:
def load_benchmark_files(directory_path, filter_prefix=None):
    loaded_instances = {}

    # Verify directory exists
    if not os.path.exists(directory_path):
        print(f"directory not found at {directory_path}")
        return {}

    files = sorted([f for f in os.listdir(directory_path) if f.endswith('.txt')])

    # Filter only 'ta' or 'dmu' files
    if filter_prefix:
        files = [f for f in files if f.lower().startswith(filter_prefix.lower())]

    print(f"Found {len(files)} benchmark files")

    for file_name in files:
        full_path = os.path.join(directory_path, file_name)
        try:
            with open(full_path, 'r') as f:
                loaded_instances[file_name] = f.read()
        except Exception as e:
            print(f"Could not read {file_name}: {e}")

    return loaded_instances

def load_solutions(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    lookup = {}
    for entry in data:
        name = entry['name'].lower() # e.g. "ta01"
        # Use optimum if available, else lower bound
        best = entry.get('optimum')
        if best is None and entry.get('bounds'):
            best = entry['bounds']['lower']
        lookup[name] = best
    return lookup

In [ ]:
# 1. Path variables
ta_folder_path = 'benchmarks/ta'
dmu_folder_path = 'benchmarks/dmu'
classical_path = 'benchmarks/classical.json'

# 2. Load the benchmark instances
print("Loading Taillard Instances")
ta_raw_data = load_benchmark_files(ta_folder_path, filter_prefix='ta')

print("\nLoading DMU Instances")
dmu_raw_data = load_benchmark_files(dmu_folder_path, filter_prefix='dmu')

print("\nLoading Solutions")
solution_lookup = load_solutions(classical_path)

print(f"Loaded {len(ta_raw_data)} TA files, {len(dmu_raw_data)} DMU files.")
print(f"Loaded {len(solution_lookup)} known solutions.")

# GNN Encoding of Benchmark Instances

In [ ]:
# Convert Benchmark Instances (TA and DMU) to GNN Encoding
def parse_matrix_jssp_instance(instance: Dict) -> Dict:
    # 1. Clean and tokenize the input
    raw_text = instance.get('input', '') or instance.get('raw_text', '')

    # Remove comments and tokenize all integers
    tokens = [int(x) for x in raw_text.split() if x.isdigit() or (x.startswith('-') and x[1:].isdigit())]

    iterator = iter(tokens)

    try:
        # 2. Parse Header (Num Jobs, Num Machines)
        num_jobs = next(iterator)
        num_machines = next(iterator)

        job_operations = {}

        # We collect all remaining data to decide the format
        data_tokens = list(iterator)
        total_tokens = len(data_tokens)
        expected_tokens = num_jobs * num_machines * 2

        if total_tokens != expected_tokens:
            raise ValueError(f"Token mismatch: Expected {expected_tokens}, got {total_tokens}")

        # 3. Determine Format and Parse Accordingly
        is_interleaved = True

        # Construct the job dictionary
        # Format 1: Interleaved (Your Output) -> Row 1: M D M D...
        # Format 2: Split (Standard TA)     -> Row 1: D D D... then Row N+1: M M M...
        # Parsing Logic for Interleaved (Row by Row: M, D, M, D...)
        ops_list = []
        for j in range(num_jobs):
            current_job_ops = []
            for m in range(num_machines):
                # Calculate index in the flat token list
                base_idx = (j * num_machines * 2) + (m * 2)

                # Interleaved: tokens[base_idx] is Machine, tokens[base_idx+1] is Duration
                machine_id = data_tokens[base_idx]
                duration = data_tokens[base_idx+1]
                current_job_ops.append((machine_id, duration))
            ops_list.append(current_job_ops)

        # Sanity Check: If all machine_ids are within bounds [0, num_machines], we accept it.
        # If the 'machine_id' we parsed is actually a duration (e.g. 94), it will likely exceed bounds.
        valid_interleaved = all(0 <= op[0] < num_machines for job in ops_list for op in job)

        if valid_interleaved:
            # Case A: It matches our provided snippet
            for j, ops in enumerate(ops_list):
                job_operations[j] = ops
        else:
            # Case B: It is likely Split Format (Times Matrix then Machines Matrix)
            # First half of data_tokens = Times, Second half = Machines
            midpoint = num_jobs * num_machines
            times_tokens = data_tokens[:midpoint]
            machines_tokens = data_tokens[midpoint:]

            for j in range(num_jobs):
                current_job_ops = []
                for m in range(num_machines):
                    # In split format, the m-th element of row j is the data
                    idx = (j * num_machines) + m
                    duration = times_tokens[idx]
                    machine_id = machines_tokens[idx]

                    # Note: We found that Standard Taillard machines are 1-indexed (1..M).
                    if machine_id > 0 and machine_id <= num_machines:
                         # Likely 1-based, convert to 0-based
                         # But be careful if 0 is used. We assume 0-based if 0 exists.
                         if 0 not in machines_tokens:
                             machine_id -= 1

                    current_job_ops.append((machine_id, duration))
                job_operations[j] = current_job_ops

        # 4. Final Structure
        return {
            'num_jobs': num_jobs,
            'num_machines': num_machines,
            'num_operations': num_jobs * num_machines,
            'job_operations': job_operations,
        }

    except StopIteration:
        print("Error: File is empty or header is missing.")
        return {}
    except Exception as e:
        print(f"Error parsing instance: {e}")
        return {}

    # Number of jobs and machines are already stored in the instance, just need to parse operations
    job_operations = {}
    lines = instance['input'].strip().split('\n')
    for i in range(len(lines)):
        # Assumes 'jobs' specified every second line, with 'machine-operations' every other line
        if i % 2 == 0:
            job_id = int(re.search(r'J(\d+):', lines[i]).group(1))
            operations = re.findall(r'M(\d+):(\d+)', lines[i+1])
            job_operations[job_id] = [(int(m), int(d)) for m, d in operations]
        # Skip every other line, as we already handle them above
        else:
            continue

    # Also extract number of operations
    # NOTE: We assume that each job has the same number of operations, so we just multiply
    # the number of operations in the first job by the number of jobs
    num_operations = instance['num_jobs'] * len(next(iter(job_operations.values())))

    # Return everything as structured dictionary
    return {
        'num_jobs': instance['num_jobs'],
        'num_machines': instance['num_machines'],
        'num_operations': num_operations,
        'job_operations': job_operations,
    }

    lines = nl_description.strip().split('\n')

    # Extract number of jobs and machines
    problem_line = lines[0]
    num_jobs = int(re.search(r'(\d+)\s+Jobs', problem_line).group(1))
    num_machines = int(re.search(r'(\d+)\s+Machines', problem_line).group(1))

    # Parse operations
    job_operations = {}
    for line in lines:
        if line.startswith('J'):
            job_id = int(re.search(r'J(\d+):', line).group(1))
            operations = re.findall(r'M(\d+):(\d+)', line)
            job_operations[job_id] = [(int(m), int(d)) for m, d in operations]

    # Extract number of operations
    # NOTE: We assume that each job has the same number of operations, so we just multiply
    # the number of operations in the first job by the number of jobs
    num_operations = num_jobs * len(next(iter(job_operations.values())))

    return {
        'num_jobs': num_jobs,
        'num_machines': num_machines,
        'num_operations': num_operations,
        'job_operations': job_operations,
    }

In [ ]:
# Function to build disjunctive graph representation of the parsed (structured) JSSP dictionary
# Uses networkx.DiGraph to store the directed graph, for better interpretability
def build_disjunctive_graph(parsed_problem: Dict) -> Tuple[nx.DiGraph, Dict]:
    # Initialise the graph
    G = nx.DiGraph()

    # Extract the operations for this JSSP instance
    job_ops = parsed_problem['job_operations']

    # Add source and sink nodes
    source = ('source', 0)
    sink = ('sink', 0)
    G.add_node(source, node_type='source', processing_time=0)
    G.add_node(sink, node_type='sink', processing_time=0)

    # Initialise reference dictionariesMap operation nodes and create node info dictionary
    node_to_operation = {}  # node -> (job_id, op_id, machine_id, proc_time) [NOT NEEDED?]
    operations_by_machine = defaultdict(list)   # machine_id -> [nodes]

    # Iterate over each job to create nodes of each operation
    for job_id, operations in job_ops.items():
        # Then iterate over each operation of that job
        for op_id, (machine_id, proc_time) in enumerate(operations):
            # Each node stores the current job and operation
            node = (job_id, op_id)

            # Add the node, with relevant info for this operation
            G.add_node(node,
                      node_type='operation',
                      job_id=job_id,
                      operation_id=op_id,
                      machine_id=machine_id,
                      processing_time=proc_time)

            # Add same info to our reference dictionaries
            node_to_operation[node] = (job_id, op_id, machine_id, proc_time)
            # operation_to_node[(job_id, op_id)] = node
            operations_by_machine[machine_id].append(node)

    # Iterate over each job to add conjunctive edges (precedence constraints within jobs)
    for job_id, operations in job_ops.items():
        # First edge: Source → first operation of this job
        # first_op = operation_to_node[(job_id, 0)]
        first_op = (job_id, 0)
        G.add_edge(source, first_op, edge_type='conjunctive', weight=0)

        # Main edges: Operations in sequence within this job
        for op_id in range(len(operations) - 1):
            # from_node = operation_to_node[(job_id, op_id)]
            # to_node = operation_to_node[(job_id, op_id + 1)]
            from_node = (job_id, op_id)
            to_node = (job_id, op_id + 1)
            proc_time = operations[op_id][1]
            G.add_edge(from_node, to_node, edge_type='conjunctive', weight=proc_time)

        # Last edge: Last operation of this job -> sink
        # last_op = operation_to_node[(job_id, len(operations) - 1)]
        last_op = (job_id, len(operations) - 1)
        last_proc_time = operations[-1][1]
        G.add_edge(last_op, sink, edge_type='conjunctive', weight=last_proc_time)

    # Iterate over each machine to add disjunctive edges (machine constraints)
    # These will be undirected initially; orientation and weighting happens during scheduling
    for machine_id, ops_on_machine in operations_by_machine.items():
        # Initial graph is complete bipartite graph (all pairs can conflict)
        for i in range(len(ops_on_machine)):
            for j in range(i + 1, len(ops_on_machine)):
                op1, op2 = ops_on_machine[i], ops_on_machine[j]
                # Add as undirected edge (will be oriented during solution)
                G.add_edge(op1, op2, edge_type='disjunctive_undirected', machine_id=machine_id)
                G.add_edge(op2, op1, edge_type='disjunctive_undirected', machine_id=machine_id)

    # Return initialised graph, with mappings of each node to its associated operation info
    return G, node_to_operation

In [ ]:
# Function to convert networkx.DiGraph to a PyTorch Geometric Data object
def graph_to_pygdata(G: nx.DiGraph,
                     node_to_operation: Dict) -> Data:
    # NOTE: Not currently using node_to_operation, as all the op info is stored in the node
    # Extract all operation nodes (we exclude source/sink initially)
    operation_nodes = [n for n in G.nodes() if G.nodes[n]['node_type'] == 'operation']
    num_jobs = max([G.nodes[n]['job_id'] for n in operation_nodes]) + 1
    num_machines = max([G.nodes[n]['machine_id'] for n in operation_nodes]) + 1
    max_ops_per_job = max([G.nodes[n]['operation_id'] for n in operation_nodes]) + 1
    
    # NOTE: Reordering the list to put source at start and sink at end, but this may not be needed
    # If so, can just use `node_idx = {node: i for i, node in enumerate(list(G))}`
    node_list = [list(G)[0]] + operation_nodes + [list(G)[1]]
    node_idx = {node: i for i, node in enumerate(node_list)}

    # Iterate over nodes to create node feature matrix
    num_nodes = len(node_list)
    node_features = []
    for node in node_list:
        if G.nodes[node]['node_type'] == 'source':
            # Source node: all zeros
            features = [0] * (num_machines + num_jobs + 2)
            node_features.append(features)
        elif G.nodes[node]['node_type'] == 'sink':
            # Sink node: all zeros
            features = [0] * (num_machines + num_jobs + 2)
            node_features.append(features)
        else:
            # Operation node
            machine_id = G.nodes[node]['machine_id']
            job_id = G.nodes[node]['job_id']
            op_idx = G.nodes[node]['operation_id']
            proc_time = G.nodes[node]['processing_time']

            # One-hot machine encoding
            machine_one_hot = [1 if i == machine_id else 0 for i in range(num_machines)]

            # One-hot job encoding
            job_one_hot = [1 if i == job_id else 0 for i in range(num_jobs)]

            # Normalized operation index and processing time
            op_idx_norm = op_idx / max(1, max_ops_per_job - 1)
            proc_time_norm = proc_time / 500.0  # Normalize by max typical processing time

            features = machine_one_hot + job_one_hot + [op_idx_norm, proc_time_norm]
            node_features.append(features)

    x = torch.tensor(node_features, dtype=torch.float32)

    # Create edge list
    edge_index = [[], []]
    edge_attr = []

    for u, v, data in G.edges(data=True):
        u_idx = node_idx[u]
        v_idx = node_idx[v]
        edge_index[0].append(u_idx)
        edge_index[1].append(v_idx)

        # Edge type: conjunctive=1, disjunctive=0
        edge_type = 1 if data['edge_type'] == 'conjunctive' else 0
        weight = data.get('weight', 0) / 500.0  # Normalize weight

        edge_attr.append([edge_type, weight])

    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.zeros((0, 2))

    # Create PyTorch Geometric Data object
    pyg_data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        num_nodes=num_nodes,
        num_jobs=num_jobs,
        num_machines=num_machines,
        node_mapping=node_idx
    )

    return pyg_data

In [ ]:
# Use above utility functions to convert natural language grompt into PyG Data
from pprint import pp

def convert_benchmark_instance_to_gnn(instance):
    # 1. Handle String Input automatically
    if isinstance(instance, str):
        instance = {'input': instance, 'filename': 'unknown'}

    # Step 1: Parse using the Matrix Parser
    parsed = parse_matrix_jssp_instance(instance)

    if not parsed or not parsed.get('job_operations'):
        print(f"skip conversion for empty/invalid instance.")
        return None

    G, node_to_op = build_disjunctive_graph(parsed)
    pyg_data = graph_to_pygdata(G, node_to_op)

    if 'filename' in instance:
        pyg_data.filename = instance['filename']

    return pyg_data

In [ ]:
def process_benchmark_dataset(raw_data_dict, dataset_name="Dataset"):
    gnn_dataset = []
    
    # Iterate over items to get filename AND content
    for filename, raw_text in tqdm(raw_data_dict.items(), desc=f"Processing {dataset_name}"):
        try:
            # A. Prepare the instance dictionary
            instance_dict = {
                'input': raw_text,
                'filename': filename
            }

            # B. Convert to GNN
            pyg_data = convert_benchmark_instance_to_gnn(instance_dict)

            if pyg_data is None:
                continue

            # C. Attach Target Solution (Lookup from classical.json)
            # Remove extension to match key (e.g., "ta01.txt" -> "ta01")
            name_key = os.path.splitext(filename)[0].lower()

            if name_key in solution_lookup:
                pyg_data.target_solution = solution_lookup[name_key]
                pyg_data.has_solution = True
            else:
                # If we don't have a known solution, we set to None or -1
                pyg_data.target_solution = -1
                pyg_data.has_solution = False

            gnn_dataset.append(pyg_data)

        except Exception as e:
            print(f"failed to parse {filename}: {e}")
            continue
    return gnn_dataset

In [ ]:
TA_dataset_gnn = process_benchmark_dataset(ta_raw_data, "TA")
DMU_dataset_gnn = process_benchmark_dataset(dmu_raw_data, "DMU")

print(f"Created TA {len(TA_dataset_gnn)} GNN instances")
print(f"Created DMU {len(DMU_dataset_gnn)} GNN instances")

# Verify one sample
if len(TA_dataset_gnn) > 0:
    sample = TA_dataset_gnn[0]
    print(f"\Sample Check ({sample.filename}):")
    print(f"Target Solution: {getattr(sample, 'target_solution', 'Not Found')}")
    print(f"Nodes: {sample.num_nodes}")

## Saving the dataset

In [ ]:
torch.save(TA_dataset_gnn, "benchmarks/gnn_dataset/TA_dataset_gnn.pt")
torch.save(DMU_dataset_gnn, "benchmarks/gnn_dataset/DMU_dataset_gnn.pt")

# GNN To NL

In [ ]:
def serialize_pyg_constraints_rules(pyg_data: Data, original_instance: Dict = None) -> str:
    num_jobs = pyg_data.num_jobs
    num_machines = pyg_data.num_machines

    # Extract structure
    nodes_by_job = defaultdict(list)
    operations_by_machine = defaultdict(list)
    node_features_map = {}

    for node_idx in range(1, pyg_data.num_nodes - 1):
        x_features = pyg_data.x[node_idx]
        machine_id = torch.argmax(x_features[:num_machines]).item()
        job_id = torch.argmax(x_features[num_machines:num_machines+num_jobs]).item()

        op_idx = int(x_features[-2].item() * 10)
        proc_time = int(x_features[-1].item() * 500)

        nodes_by_job[job_id].append((op_idx, node_idx, machine_id, proc_time))
        operations_by_machine[machine_id].append((job_id, op_idx, node_idx, proc_time))
        node_features_map[node_idx] = (job_id, op_idx, machine_id, proc_time)

    lines = ["## CONSTRAINT SPECIFICATION FOR JSSP\n"]

    # 1. Job precedence constraints (ALL OF THEM)
    lines.append("### JOB PRECEDENCE CONSTRAINTS")
    lines.append("(Each job's operations must execute in fixed order)\n")
    precedence_seen = set()  # Deduplication

    for job_id in sorted(nodes_by_job.keys()):
        ops = sorted(nodes_by_job[job_id], key=lambda x: x[0])
        op_names = [f"J{job_id}_Op{op_idx}" for op_idx, _, _, _ in ops]

        if len(op_names) == 0:
            continue

        # Single chain line for readability
        chain_str = " -> ".join(op_names)
        lines.append(f"Job {job_id}: {chain_str}")

        # If hardware is strong enough, also emit full explicit precedence rules
        # for i in range(len(op_names) - 1):
        #     constraint_pair = (op_names[i], op_names[i+1])
        #     if constraint_pair not in precedence_seen:
        #         lines.append(f"MUST_PRECEDE: {op_names[i]} -> {op_names[i+1]}")
        #         lines.append(f"  Constraint: {op_names[i]} must complete before {op_names[i+1]} starts")
        #         precedence_seen.add(constraint_pair)

        lines.append("")  # Blank line between jobs

    # 2. Machine exclusivity constraints (ALL OF THEM)
    lines.append("\n### MACHINE EXCLUSIVITY CONSTRAINTS")
    lines.append("(Operations on same machine cannot overlap)\n")

    mutex_seen = set()  # Deduplication

    for machine_id in sorted(operations_by_machine.keys()):
        # ops = operations_by_machine[machine_id]
        ops = sorted(operations_by_machine[machine_id], key=lambda x: (x[0], x[1]))  # Sort by job, op_idx
        op_names = [f"J{job_id}_Op{op_idx}" for job_id, op_idx, _, _ in ops]

        if len(op_names) == 0:
            continue

        lines.append(f"MACHINE {machine_id} OPERATIONS: {{{', '.join(op_names)}}}")

        # If hardware is strong enough, also generate all pairwise exclusivity constraints
        # for i in range(len(ops)):
        #     for j in range(i + 1, len(ops)):
        #         op1_name = op_names[i]
        #         op2_name = op_names[j]

        #         constraint_pair = tuple(sorted([op1_name, op2_name]))
        #         if constraint_pair not in mutex_seen:
        #             lines.append(f"  MUTEX: {op1_name} (+) {op2_name}")
        #             mutex_seen.add(constraint_pair)

        # Else, put in a generic instruction to obey exclusivity constraints
        lines.append(
            "  Rule: Operations in this set must NOT execute in parallel; "
            "they must be totally ordered in time."
        )

        lines.append("")

    # 3. Schedule feasibility rules
    lines.append("\n### SCHEDULING RULES")
    lines.append("To generate a valid schedule, you MUST:")
    lines.append("  1. Respect ALL precedence constraints ( -> )")
    lines.append("  2. Ensure NO machine has overlapping operations ( (+) )")
    lines.append("  3. Assign start times to each operation based on these constraints")
    lines.append("  4. Minimize the maximum end time (makespan)\n")

    # 4. Summary statistics
    lines.append("### CONSTRAINT SUMMARY")
    # total_precedence = sum(len(ops) - 1 for ops in nodes_by_job.values()) + num_jobs
    total_precedence = sum(len(ops) - 1 for ops in nodes_by_job.values())
    total_mutex = sum(len(ops) * (len(ops) - 1) // 2 for ops in operations_by_machine.values())
    lines.append(f"Total precedence constraints: {total_precedence}")
    lines.append(f"Total machine exclusivity constraints: {total_mutex}")
    lines.append(f"Total constraints: {total_precedence + total_mutex}\n")

    return "\n".join(lines)

# Evaluation

In [ ]:
model_dir = "model"
TRAINED_MODEL_PATH = os.path.join(model_dir, "trained_model")

#Loading Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = TRAINED_MODEL_PATH,
    max_seq_length = 4096,
    dtype = None,           # Auto-detects based on your GPU (A6000 uses bfloat16)
    load_in_4bit = True,    # Matches your training quantization
)

#Inference Mode
FastLanguageModel.for_inference(model)
print("Model loaded successfully! Ready for GNN-to-Text inference.")

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def convert_benchmark_gnn_to_inference_format(gnn_dataset, alpaca_prompt):
    texts = []
    metadata = [] # To track filename and target for scoring later

    print(f"converting {len(gnn_dataset)} instances...")

    for gnn_item in tqdm(gnn_dataset):
        # 1. Standardized Instruction (Matches your training code)
        instruction = f"Optimize schedule for {gnn_item.num_jobs} Jobs (denoted as J) across {gnn_item.num_machines} Machines (denoted as M) to minimize makespan. The makespan is the completion time of the last operation in the schedule. Each M can process only one J at a time, and once started, J cannot be interrupted."

        # 2. Serialize Graph (Input)
        # We pass None for original_nl because benchmarks don't have it
        graph_representation = serialize_pyg_constraints_rules(gnn_item)

        # 3. Format Prompt (Empty Output for Inference)
        # We use empty string "" for the solution part so the model generates it
        formatted = alpaca_prompt.format(
            instruction,
            graph_representation,
            "" # <--- EMPTY for Inference!
        ) + EOS_TOKEN

        texts.append(formatted)

        # Store metadata for the evaluator
        metadata.append({
            "filename": getattr(gnn_item, 'filename', 'unknown'),
            "target": getattr(gnn_item, 'target_solution', -1)
        })

    # Return HF Dataset with metadata columns
    return Dataset.from_dict({
        'text': texts,
        'filename': [m['filename'] for m in metadata],
        'target_solution': [m['target'] for m in metadata],
        'idx': list(range(len(texts)))
    })

print("Converting GNN Benchmarks to Text Format for Inference...")
# Define Alpaca Prompt (Ensure placeholders match .format args)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Convert TA
TA_inference_dataset = convert_benchmark_gnn_to_inference_format(
    gnn_dataset=TA_dataset_gnn,
    alpaca_prompt=alpaca_prompt
)

# Convert DMU
DMU_inference_dataset = convert_benchmark_gnn_to_inference_format(
    gnn_dataset=DMU_dataset_gnn,
    alpaca_prompt=alpaca_prompt
)

print(f"TA inference Dataset: {len(TA_inference_dataset)} instances")
print(f"DMU inference Dataset: {len(DMU_inference_dataset)} instances")

print("\n Sample inference Prompt")
print(TA_inference_dataset[0]['text'][:500])

In [ ]:
TA_inference_dataset.to_json('benchmarks/gnn_dataset/TA_inference_dataset.json')
print("Task Done")
DMU_inference_dataset.to_json('benchmarks/gnn_dataset/DMU_inference_dataset.json')
print("Task Done")

# Comparisons

Couldn't run these cells as the output from above cells was unusable.

In [ ]:
from datasets import load_dataset
import pandas as pd
eval_dataset = load_dataset("json", data_files="gnn_dataset/eval_dataset_gnn_text.json")["train"]

In [ ]:
def run_benchmark_evaluation(model, tokenizer, dataset, name):
    results = []
    print(f"Running Inference on {name} ({len(dataset)} samples)...")

    for i in tqdm(range(len(dataset))):
        prompt = dataset[i]['text']
        target = 0
        target = dataset[i]['target_solution']
        fname = dataset[i]['filename']

        # 1. Generate response
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True)
        decoded = tokenizer.batch_decode(outputs)
        print(decoded)

        # 2. Extract Makespan (Maximum completion time)
        # We look for the "-> [TIME]" pattern used in your training format
        times = re.findall(r"->\s*(\d+)", decoded)
        prediction = max(map(int, times)) if times else -1

        # 3. Calculate Gap (%)
        # Optimality Gap = (Predicted - Optimal) / Optimal * 100
        gap = ((prediction - target) / target * 100) if (target > 0 and prediction > 0) else 100.0

        results.append({
            "Instance": fname.replace(".txt", ""),
            "Optimal": target,
            "Predicted": prediction,
            "Gap %": gap
        })

        if i > 1:
            break

    return pd.DataFrame(results)

# df_ta = run_benchmark_evaluation(model, tokenizer, eval_dataset, "Evaluation")
df_dmu = run_benchmark_evaluation(model, tokenizer, DMU_inference_dataset, "DMU")
df_ta = run_benchmark_evaluation(model, tokenizer, TA_inference_dataset, "Taillard")

# Save raw results for your report appendix
df_ta.to_csv("TA_Results.csv", index=False)
df_dmu.to_csv("DMU_Results.csv", index=False)